In [1]:
import pandas as pd

# Memuat data training
train_path = '../data/processed/train.csv'
df_train = pd.read_csv(train_path)

print(f"Total data latih: {len(df_train)} baris")
display(df_train.head())

Total data latih: 80000 baris


,user_id,item_id,rating,timestamp
0,259,255,4,1997-09-20 03:05:10
1,259,286,4,1997-09-20 03:05:27
2,259,298,4,1997-09-20 03:05:54
3,259,185,4,1997-09-20 03:06:21
4,259,173,4,1997-09-20 03:07:23


In [2]:
# Menghitung metrik popularitas untuk setiap item
# Kita gunakan agg() untuk menghitung total interaksi (count) dan rata-rata rating (mean)
item_popularity = df_train.groupby('item_id').agg(
    total_interactions=('rating', 'count'),
    avg_rating=('rating', 'mean')
).reset_index()

# Aturan bisnis (Business Rule) sederhana:
# Jangan merekomendasikan item yang baru di-klik 1-2 kali tapi ratingnya kebetulan 5.
# Kita filter minimal harus ada 50 interaksi agar valid, lalu urutkan berdasarkan interaksi terbanyak.
popular_items = item_popularity[item_popularity['total_interactions'] >= 50].copy()
popular_items = popular_items.sort_values('total_interactions', ascending=False)

print("Top 5 Item Paling Populer di Platform:")
display(popular_items.head(5))

Top 5 Item Paling Populer di Platform:


,item_id,total_interactions,avg_rating
49,50,473,4.355180
180,181,423,4.000000
99,100,417,4.153477
293,294,396,3.146465
257,258,394,3.817259


In [3]:
def get_popular_recommendations(user_id, top_n=5):
    """
    Fungsi untuk memberikan rekomendasi berbasis popularitas.
    Perhatikan bahwa parameter user_id tidak mengubah hasil akhir, 
    karena model ini belum dipersonalisasi.
    """
    # Ambil top N item
    recommendations = popular_items.head(top_n)[['item_id', 'total_interactions']]
    
    # Format output menjadi list of dictionaries (Mirip format JSON API)
    result = []
    for _, row in recommendations.iterrows():
        result.append({
            "item_id": int(row['item_id']),
            "score": int(row['total_interactions']) # Skor menggunakan jumlah interaksi
        })
        
    return {
        "user_id": user_id,
        "recommendation_type": "popularity_baseline",
        "recommendations": result
    }

# Mari kita tes!
# Meminta rekomendasi untuk User ID 999 dan User ID 12
print("Rekomendasi untuk User 999:")
print(get_popular_recommendations(user_id=999, top_n=3))

print("\nRekomendasi untuk User 12:")
print(get_popular_recommendations(user_id=12, top_n=3))

Rekomendasi untuk User 999:
{'user_id': 999, 'recommendation_type': 'popularity_baseline', 'recommendations': [{'item_id': 50, 'score': 473}, {'item_id': 181, 'score': 423}, {'item_id': 100, 'score': 417}]}

Rekomendasi untuk User 12:
{'user_id': 12, 'recommendation_type': 'popularity_baseline', 'recommendations': [{'item_id': 50, 'score': 473}, {'item_id': 181, 'score': 423}, {'item_id': 100, 'score': 417}]}
